In [7]:
import numpy as np
from system import System, inverse_power_potential, kinetic

In [8]:
def upward_section_returns(q, p, h, y_section=5.0):
    """Interpolate crossings of y1=y_section with p_y1 > 0."""
    signed_distance = q[:, 0, 1] - y_section
    crossing_steps = np.flatnonzero(
        (signed_distance[:-1] < 0.0)
        & (signed_distance[1:] >= 0.0)
    )

    times = []
    q_returns = []
    p_returns = []
    for step in crossing_steps:
        fraction = -signed_distance[step] / (
            signed_distance[step + 1] - signed_distance[step]
        )
        q_crossing = q[step] + fraction * (q[step + 1] - q[step])
        p_crossing = p[step] + fraction * (p[step + 1] - p[step])

        if p_crossing[0, 1] > 0.0:
            times.append((step + fraction) * h)
            q_returns.append(q_crossing)
            p_returns.append(p_crossing)

    return np.asarray(times), np.asarray(q_returns), np.asarray(p_returns)


x_star = 3.8351644331875563
q0 = np.array([
    [x_star, 5.0],
    [10.0 - x_star, 5.0],
])
p0 = np.array([
    [0.0, 20.0],
    [0.0, -20.0],
])

system = System(
    # System evolves its NumPy inputs in place, so preserve q0 and p0 for
    # the fixed-point residual below.
    q0=q0.copy(),
    p0=p0.copy(),
    KE=kinetic,
    V=inverse_power_potential,
    V_parameters={
        "k": 1.0,
        "alpha": 1.0,
        "beta": 1.0,
        "C": 1.0,
        "Lx": 10.0,
        "Ly": 10.0,
    },
    masses=(1.0, 1.0),
)

In [9]:
h = 0.0001
nsteps = 100000
q, p, energy = system.leapfrog(h=h, nsteps=nsteps)
times, q_returns, p_returns = upward_section_returns(q, p, h)

print(f"number of upward returns: {len(times)}")
print(f"max |E(t)-E(0)|: {np.max(np.abs(energy - energy[0])):.3e}")
print(
    "max relative energy error: "
    f"{np.max(np.abs((energy - energy[0]) / energy[0])):.3e}"
)
print("return       time        max|dq|       max|dp|       ||delta z||_2")
for number, (time, q_return, p_return) in enumerate(
    zip(times, q_returns, p_returns),
    start=1,
):
    dq = q_return - q0
    dp = p_return - p0
    state_residual = np.linalg.norm(
        np.concatenate((dq.ravel(), dp.ravel()))
    )
    print(
        f"{number:6d}  {time:11.7f}  "
        f"{np.max(np.abs(dq)):12.3e}  "
        f"{np.max(np.abs(dp)):12.3e}  "
        f"{state_residual:16.3e}"
    )

if len(times):
    print(f"\nfirst-return period estimate: T = {times[0]:.10f}")
    print("first Poincare point q:\n", q_returns[0])
    print("first Poincare point p:\n", p_returns[0])

number of upward returns: 9
max |E(t)-E(0)|: 1.388e+00
max relative energy error: 3.452e-03
return       time        max|dq|       max|dp|       ||delta z||_2
     1    1.0027085     1.777e-07     3.514e-07         5.570e-07
     2    2.0054170     6.962e-07     6.742e-07         1.371e-06
     3    3.0081256     1.513e-06     9.420e-07         2.521e-06
     4    4.0108341     2.562e-06     1.133e-06         3.962e-06
     5    5.0135426     3.757e-06     1.232e-06         5.592e-06
     6    6.0162511     5.002e-06     1.230e-06         7.284e-06
     7    7.0189596     6.193e-06     1.128e-06         8.903e-06
     8    8.0216682     7.235e-06     9.336e-07         1.032e-05
     9    9.0243767     8.042e-06     6.633e-07         1.141e-05

first-return period estimate: T = 1.0027085198
first Poincare point q:
 [[3.83516426 5.        ]
 [6.16483574 5.        ]]
first Poincare point p:
 [[-3.51443896e-07  2.00000000e+01]
 [ 3.51443896e-07 -2.00000000e+01]]


In [10]:
SECTION_COORDINATES = (
    "x1", "x2", "y2", "px1", "py1", "px2", "py2",
)


def pack_section_state(q, p):
    """Remove y1, which is fixed by the section y1=5."""
    return np.array([
        q[0, 0], q[1, 0], q[1, 1],
        p[0, 0], p[0, 1], p[1, 0], p[1, 1],
    ])


def unpack_section_state(z, y_section=5.0):
    q = np.array([
        [z[0], y_section],
        [z[1], z[2]],
    ])
    p = np.array([
        [z[3], z[4]],
        [z[5], z[6]],
    ])
    return q, p


def first_upward_return(force, masses, q0, p0, h, max_steps=50000):
    """Leapfrog to the next y1=5, py1>0 crossing and interpolate it."""
    q = q0.copy()
    p = p0.copy()
    y_section = q0[0, 1]

    for step in range(max_steps):
        q_before = q.copy()
        p_before = p.copy()
        distance_before = q_before[0, 1] - y_section

        p_half = p_before + 0.5 * h * force(q_before)
        q = q_before + h * p_half / masses[:, None]
        p = p_half + 0.5 * h * force(q)
        distance_after = q[0, 1] - y_section

        if distance_before < 0.0 <= distance_after:
            fraction = -distance_before / (
                distance_after - distance_before
            )
            q_return = q_before + fraction * (q - q_before)
            p_return = p_before + fraction * (p - p_before)
            if p_return[0, 1] > 0.0:
                return (step + fraction) * h, q_return, p_return

    raise RuntimeError("No upward Poincare return found within max_steps")


def poincare_map(z, force, masses, h):
    q0, p0 = unpack_section_state(z)
    _, q_return, p_return = first_upward_return(
        force, masses, q0, p0, h,
    )
    return pack_section_state(q_return, p_return)


def poincare_jacobian(z_star, force, masses, h, relative_step):
    """Central-difference approximation to DP(z_star) on the section."""
    dimension = len(z_star)
    jacobian = np.empty((dimension, dimension))

    for column in range(dimension):
        delta = relative_step * max(1.0, abs(z_star[column]))
        z_plus = z_star.copy()
        z_minus = z_star.copy()
        z_plus[column] += delta
        z_minus[column] -= delta

        jacobian[:, column] = (
            poincare_map(z_plus, force, masses, h)
            - poincare_map(z_minus, force, masses, h)
        ) / (2.0 * delta)

    return jacobian

In [11]:
h = 0.0001
relative_step = 1e-6

x_star = 3.8351644331875563
q_star = np.array([
    [x_star, 5.0],
    [10.0 - x_star, 5.0],
])
p_star = np.array([
    [0.0, 20.0],
    [0.0, -20.0],
])

system = System(
    q0=q_star.copy(),
    p0=p_star.copy(),
    KE=kinetic,
    V=inverse_power_potential,
    V_parameters={
        "k": 1.0,
        "alpha": 1.0,
        "beta": 1.0,
        "C": 1.0,
        "Lx": 10.0,
        "Ly": 10.0,
    },
    masses=(1.0, 1.0),
)

z_star = pack_section_state(q_star, p_star)
period, q_return, p_return = first_upward_return(
    system.force, system.masses, q_star, p_star, h,
)
fixed_point_residual = np.linalg.norm(
    pack_section_state(q_return, p_return) - z_star
)
jacobian = poincare_jacobian(
    z_star, system.force, system.masses, h, relative_step,
)
eigenvalues = np.linalg.eigvals(jacobian)
eigenvalues = eigenvalues[np.argsort(np.abs(eigenvalues))[::-1]]

print(f"section coordinates: {SECTION_COORDINATES}")
print(f"h: {h:.1e}")
print(f"relative finite-difference step: {relative_step:.1e}")
print(f"period: {period:.10f}")
print(f"fixed-point residual on section: {fixed_point_residual:.3e}")
print("\nDP(z*):")
print(np.array2string(jacobian, precision=7, suppress_small=True))
print("\nFloquet multipliers:")
for eigenvalue in eigenvalues:
    print(
        f"  {eigenvalue.real:+.10f} {eigenvalue.imag:+.10f}j"
        f"    |lambda|={abs(eigenvalue):.10f}"
    )

jacobian, eigenvalues

section coordinates: ('x1', 'x2', 'y2', 'px1', 'py1', 'px2', 'py2')
h: 1.0e-04
relative finite-difference step: 1.0e-06
period: 1.0027085198
fixed-point residual on section: 5.570e-07

DP(z*):
[[ 0.9685935  0.0093673 -0.00353    0.9928381 -0.0000024  0.0024932
   0.0000027]
 [ 0.0093673  0.9685935  0.00353    0.0024932  0.0000024  0.992838
  -0.0000027]
 [ 0.         0.0000001  0.9990616  0.0000007  1.0073452 -0.0000007
   1.0073451]
 [-0.0622305  0.0184334  0.000142   0.9685936  0.0035851  0.0093673
   0.003596 ]
 [ 0.0001451 -0.0001453 -0.000906  -0.0035285  0.9995131  0.0035285
  -0.0004694]
 [ 0.0184334 -0.0622305 -0.000142   0.0093673 -0.0035851  0.9685936
  -0.003596 ]
 [-0.0001448  0.0001457 -0.0009686  0.0035286 -0.0004641 -0.0035286
   0.9995184]]

Floquet multipliers:
  +0.9779608385 -0.2087884048j    |lambda|=0.9999999998
  +0.9779608385 +0.2087884048j    |lambda|=0.9999999998
  +0.9592262614 -0.2826393026j    |lambda|=0.9999999980
  +0.9592262614 +0.2826393026j    |lambda|=

(array([[ 9.68593530e-01,  9.36730736e-03, -3.52996219e-03,
          9.92838053e-01, -2.42484921e-06,  2.49318144e-03,
          2.74044121e-06],
        [ 9.36730925e-03,  9.68593530e-01,  3.52995935e-03,
          2.49318655e-03,  2.42508236e-06,  9.92838016e-01,
         -2.74067435e-06],
        [ 1.91060177e-08,  8.55785974e-08,  9.99061618e-01,
          6.94111435e-07,  1.00734517e+00, -6.94111435e-07,
          1.00734515e+00],
        [-6.22304752e-02,  1.84333987e-02,  1.41985319e-04,
          9.68593553e-01,  3.58514326e-03,  9.36728438e-03,
          3.59597895e-03],
        [ 1.45139964e-04, -1.45315052e-04, -9.05974673e-04,
         -3.52851792e-03,  9.99513150e-01,  3.52851792e-03,
         -4.69412331e-04],
        [ 1.84333982e-02, -6.22304756e-02, -1.41985220e-04,
          9.36728503e-03, -3.58514330e-03,  9.68593555e-01,
         -3.59597891e-03],
        [-1.44838899e-04,  1.45716436e-04, -9.68574199e-04,
          3.52859786e-03, -4.64097383e-04, -3.52859786e-03